# 1. Compreensão do Negócio

O IBGE realizou em 2022 o Censo Demográfico brasileiro.

A população dos municípios brasileiros apresenta grande variação.
Existem municípios com poucos milhares de habitantes e municípios
com populações superiores a um milhão de habitantes.

Neste exercício queremos utilizar dados reais do Censo Demográfico
2022 para construir um modelo capaz de classificar municípios de
acordo com seu porte populacional.

## Pergunta do projeto

É possível utilizar a população residente de um município para
classificá-lo de acordo com seu porte populacional utilizando KNN?

## Objetivo

1. selecionar 10 municípios;
2. obter seus dados através da API REST do IBGE;
3. construir uma tabela utilizando Pandas;
4. definir classes de porte populacional;
5. treinar um modelo KNN;
6. utilizar o modelo para classificar municípios.

# 2. Compreensão dos Dados

A fonte utilizada será a API de Agregados do IBGE.

Os dados são provenientes do Censo Demográfico 2022.

A API retorna os dados no formato JSON.

Exemplo de consulta:

https://servicodados.ibge.gov.br/api/v3/agregados/202/periodos/-6/variaveis/93|1000093?localidades=N6[3507506]&classificacao=2[0]|1[1]

Nesta etapa devemos investigar:

- como funciona a API;
- qual é a estrutura do JSON;
- onde está o código do município;
- onde está o nome do município;
- onde está o ano;
- onde está o valor da população.

## 2.1 Importar Bibliotecas

In [ ]:
import requests
import pandas as pd

## 2.2 Primeira Consulta a API

A API de Dados do IBGE pode ser acessada no enedereço [https://servicodados.ibge.gov.br/](https://servicodados.ibge.gov.br/).

Vamos consultar as tabelas (AGREGADOS) do IBGE. Vamos acessar as tabelas do CENSO Demográfico 2022.
Primeiro vamos fazer um teste [https://servicodados.ibge.gov.br/api/docs/agregados?versao=3](https://servicodados.ibge.gov.br/api/docs/agregados?versao=3)


In [ ]:
codigo_municipio = 3507506

url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/202/"
    "periodos/-6/"
    "variaveis/93|1000093"
    f"?localidades=N6[{codigo_municipio}]"
    "&classificacao=2[0]|1[1]"
)

resposta = requests.get(url)

print(resposta.status_code)

## 2.3 Converta os dados da variável `resposta` para JSON



In [ ]:
dados = resposta.json()

dados

## 2.4 Investigue a estrutura interna da variável `dados`

In [ ]:
type(dados)

In [ ]:
len(dados)

In [ ]:
dados[0]

In [ ]:
dados[0].keys()

### Atividade 1

Investigue o JSON retornado pela API.

Identifique:

1. qual é o nome da variável;
2. qual é o município;
3. qual é o código do município;
4. qual é o período da observação;
5. onde está armazenado o valor da população.

Não prossiga diretamente para a modelagem.

Primeiro procure compreender como os dados foram organizados pela API.

# 3. Preparação dos Dados

Uma resposta JSON não está necessariamente organizada no formato
necessário para utilizar algoritmos de Machine Learning.

Precisamos transformar os dados obtidos da API em uma tabela.

Vamos selecionar 10 municípios do Estado de São Paulo.

## 3.1 - Seleção dos municípios


In [ ]:
municipios = {
    "São Paulo": 3550308,
    "Campinas": 3509502,
    "Santos": 3548500,
    "Sorocaba": 3552205,
    "Ribeirão Preto": 3543402,
    "São José dos Campos": 3549904,
    "Bauru": 3506003,
    "Jundiaí": 3525904,
    "Piracicaba": 3538709,
    "Presidente Prudente": 3541406
}

Cada município será uma observação da nossa base de dados.

Portanto:

- linha → município;
- coluna → característica ou variável do município.

## 3.2 - Agora carregue a coluna "população" para cada um destes 10 municípios (10 entradas ou)

In [ ]:
resultados = []

for nome_municipio, codigo_municipio in municipios.items():

    url = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/202/"
        "periodos/-6/"
        "variaveis/93|1000093"
        f"?localidades=N6[{codigo_municipio}]"
        "&classificacao=2[0]|1[1]"
    )

    resposta = requests.get(url)

    dados = resposta.json()

    print(nome_municipio)
    print(dados)

### Atividade 2

Modifique o código anterior para recuperar o valor da população
de cada município.

Para cada município, crie um registro com a seguinte estrutura:

codigo
municipio
ano
populacao

In [ ]:
registro = {
    "codigo": codigo_municipio,
    "municipio": nome_municipio,
    "ano": 2022,
    "populacao": valor
}

## 3.3 Construção do DataFrame (sua tabela Python)

In [ ]:
df = pd.DataFrame(resultados)

df

## 3.4 Verificar os tipos dos dados

In [ ]:
df.info()

### 3.4.1 Verificar dados ausentes

In [ ]:
df.isnull().sum()

## 3.5  Mostrar Estatística Básica dos dados.

In [ ]:
df["populacao"].describe()

## 3.5 Criando as classes de porte populacional

Pequena → menos de 200.000 habitantes

Média → entre 200.000 e 500.000 habitantes

Grande → mais de 500.000 habitantes

In [ ]:
def classificar_porte(populacao):

    if populacao < 200000:
        return "Pequena"

    elif populacao <= 500000:
        return "Média"

    else:
        return "Grande"

## 3.6 Colocando tudo dentro de uma tabela python (DataFrame)

In [ ]:
df["porte"] = df["populacao"].apply(classificar_porte)

df

### Atividade 3

Quantos municípios ficaram em cada classe?

In [ ]:
df["porte"].value_counts()

# 4. Modelagem

Utilizaremos o algoritmo:

K-Nearest Neighbors — KNN

O KNN compara uma nova observação com observações já conhecidas.

Neste exemplo, a característica utilizada será:

X = população

e a variável que queremos prever será:

y = porte populacional

## 4.1 Definir X e y

In [ ]:
X = df[["populacao"]]

y = df["porte"]

Liste o conteúdo da entrada "X"

In [ ]:
X

Liste o conteúdo da saída "Y"

In [ ]:
y

## 4.2 Separar treino e teste

Deixe 70% dos dados para treino e 30% para teste

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

Visualize os dados de treino

In [ ]:
X_treino

Visualize os dados de teste

In [ ]:
X_teste

## 4.3 - Crie um modelo de Machine Learning KNN (K vizinhos mais próximos)

Use esse tipo de modelo para "agrupar" as cidades por tamanho da população

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modelo = KNeighborsClassifier(n_neighbors=3)

## 4.4 Treinar seu modelo

Você já tem seus dados de entrada e saída. Agora treine seu modelo KNN.
Se modelo está na variável `modelo`

In [ ]:
modelo.fit(X_treino, y_treino)

OBS: Lembre-se que o método "fit()" é responsável por fazer o treinamento

# 5. Avaliação ( etapa de Teste)

Na etapa de avaliação, verificamos se o modelo apresenta desempenho adequado nos dados que não participaram do treinamento.

In [ ]:
y_pred = modelo.predict(X_teste)

y_pred

Lembre-se que o método "predict()" faz o teste (ou seja, prevê valores de saída para uma entrada dada ao seu modelo).

## 5.1 Comparando o resultado real com a sída que seu modelo calculou (previu).

In [ ]:
resultado = pd.DataFrame({
    "Real": y_teste,
    "Previsto": y_pred
})

resultado

## 5.2 Calcular acurácia (acertividade) do seu modelo

In [ ]:
from sklearn.metrics import accuracy_score

acuracia = accuracy_score(y_teste, y_pred)

print("Acurácia:", acuracia)

### Atividade 4

Analise o resultado.

1. Quantos municípios foram utilizados para treinamento?
2. Quantos municípios foram utilizados para teste?
3. Quantos municípios foram classificados corretamente?
4. Qual foi a acurácia?
5. O conjunto de apenas 10 municípios é suficiente para avaliar
   adequadamente o modelo?
6. O que poderia ser feito para melhorar o experimento?

# 6. Implantação

Transforme seu modelo em um software web